# DSAI 490 — Assignment 1: Representation Learning with Autoencoders

This notebook documents the full experiment pipeline for training and analyzing
Autoencoder (AE) and Variational Autoencoder (VAE) models on the Medical MNIST dataset.

**Dataset:** Medical MNIST — 6 anatomical regions, 64×64 grayscale images  
**Models:** One AE + one VAE per region = 12 models total  
**Latent space:** 2D for direct visualization  

## 0. Setup

In [ ]:
import os
import sys

# Make sure src/ is importable from the notebook
sys.path.append(os.path.abspath(".."))

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from src.data_processing import build_region_datasets, ANATOMICAL_REGIONS
from src.model import Autoencoder, VAE
from src.utils import (
    plot_loss_curves,
    plot_vae_loss_curves,
    plot_reconstructions,
    plot_latent_space,
    plot_generated_samples,
    plot_denoising,
)

print('TensorFlow version:', tf.__version__)
print('Regions:', ANATOMICAL_REGIONS)

## 1. Load Dataset

In [ ]:
# Load all datasets — one tf.data pipeline per region
datasets = build_region_datasets('../data/raw')

# Print summary
print(f"{'Region':<15} {'Train':>8} {'Val':>8} {'Test':>8}")
print('-' * 42)
for region in ANATOMICAL_REGIONS:
    d = datasets[region]
    print(f"{region:<15} {d['n_train']:>8} {d['n_val']:>8} {d['n_test']:>8}")

## 2. Visualize Raw Data Samples

In [ ]:
# Show one sample image from each anatomical region
fig, axes = plt.subplots(1, len(ANATOMICAL_REGIONS), figsize=(18, 3))

for ax, region in zip(axes, ANATOMICAL_REGIONS):
    for batch in datasets[region]['test'].take(1):
        img = batch[0, :, :, 0].numpy()
    ax.imshow(img, cmap='gray')
    ax.set_title(region, fontsize=10)
    ax.axis('off')

plt.suptitle('Sample Image per Anatomical Region', fontsize=13)
plt.tight_layout()
plt.savefig('../figures/dataset_samples.png', dpi=150)
plt.show()

## 3. Load Trained Models

In [ ]:
# Load all 12 saved models from models/
models = {}

for region in ANATOMICAL_REGIONS:
    ae_path  = f'../models/{region}_ae_v1.keras'
    vae_path = f'../models/{region}_vae_v1.keras'

    models[region] = {
        'ae':  tf.keras.models.load_model(ae_path),
        'vae': tf.keras.models.load_model(vae_path),
    }
    print(f'  [{region}] loaded AE + VAE')

print('\nAll 12 models loaded.')

## 4. Reconstruction Analysis

In [ ]:
# AE Reconstructions — all regions
for region in ANATOMICAL_REGIONS:
    plot_reconstructions(
        models[region]['ae'],
        datasets[region]['test'],
        region, 'AE', save=True
    )
    plt.show()

In [ ]:
# VAE Reconstructions — all regions
for region in ANATOMICAL_REGIONS:
    plot_reconstructions(
        models[region]['vae'],
        datasets[region]['test'],
        region, 'VAE', save=True
    )
    plt.show()

## 5. Latent Space Visualization

In [ ]:
# AE Latent Space
for region in ANATOMICAL_REGIONS:
    plot_latent_space(
        models[region]['ae'],
        datasets[region]['test'],
        region, 'AE', save=True
    )
    plt.show()

In [ ]:
# VAE Latent Space
for region in ANATOMICAL_REGIONS:
    plot_latent_space(
        models[region]['vae'],
        datasets[region]['test'],
        region, 'VAE', save=True
    )
    plt.show()

## 6. VAE Sample Generation

In [ ]:
# Generate new images from VAE prior N(0, I)
for region in ANATOMICAL_REGIONS:
    plot_generated_samples(
        models[region]['vae'],
        region, n_samples=16, save=True
    )
    plt.show()

## 7. Denoising

In [ ]:
# Denoising — AE and VAE side by side for each region
for region in ANATOMICAL_REGIONS:
    plot_denoising(
        models[region]['ae'],
        datasets[region]['test'],
        region, 'AE', noise_factor=0.3, save=True
    )
    plt.show()

    plot_denoising(
        models[region]['vae'],
        datasets[region]['test'],
        region, 'VAE', noise_factor=0.3, save=True
    )
    plt.show()

## 8. AE vs VAE Comparison

In [ ]:
# Compute test reconstruction MSE for AE vs VAE per region
print(f"{'Region':<15} {'AE MSE':>10} {'VAE MSE':>10}")
print('-' * 38)

for region in ANATOMICAL_REGIONS:
    ae_losses, vae_losses = [], []

    for batch in datasets[region]['test']:
        # AE
        ae_recon = models[region]['ae'](batch, training=False)
        ae_losses.append(float(tf.reduce_mean(tf.square(batch - ae_recon))))

        # VAE
        vae_recon, _, _ = models[region]['vae'](batch, training=False)
        vae_losses.append(float(tf.reduce_mean(tf.square(batch - vae_recon))))

    ae_mse  = sum(ae_losses)  / len(ae_losses)
    vae_mse = sum(vae_losses) / len(vae_losses)
    print(f"{region:<15} {ae_mse:>10.4f} {vae_mse:>10.4f}")

## 9. Key Observations

*(Fill in after running all cells)*

- **AE vs VAE reconstruction quality:** ...
- **Latent space structure:** ...
- **VAE generation quality:** ...
- **Denoising observations:** ...
- **Which regions were hardest to reconstruct and why:** ...